<a href="https://colab.research.google.com/github/kimunilaz/Data_Structures-Algorithms_Final_Project/blob/main/Data_Structures%26Algorithms_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Loading the data and building the graph**

This section reads the two input files.



*   Q2_road_network.csv gives 25 roads, each with two junctions and a travel time in minutes.
*   Q2_residential_areas.csv gives 18 residential areas and the junction where each one is located.






The road network is stored as an adjacency list: each junction maps to a list of (neighbour, travel time) pairs. Every road can be travelled in both directions, so each road is added twice, once from each end. The graph is undirected and weighted, with travel time as the weight.

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

graph = defaultdict(list)

for _, row in roads.iterrows():
    a = row["from"]
    b = row["to"]
    t = row["minutes"]
    graph[a].append((b, t))
    graph[b].append((a, t))   # roads go both ways, so add both directions

junctions = sorted(graph.keys())

s_path = "https://raw.githubusercontent.com/kimunilaz/Data_Structures-Algorithms_Final_Project/main/"

def load_csv(filename):
    if os.path.exists(filename):
        return pd.read_csv(filename)          # running locally after cloning
    return pd.read_csv(s_path + filename)     # running in Colab, reads from GitHub

roads = load_csv("Q2_road_network.csv")
areas = load_csv("Q2_residential_areas.csv")

#print(roads.head(5))
#print(areas.head(5))

#building adjacency list

graph = defaultdict(list)

for _, row in roads.iterrows():
    a = row["from"]
    b = row["to"]
    t = row["minutes"]
    graph[a].append((b, t))
    graph[b].append((a, t))   # roads go both ways, so add both directions

junctions = sorted(graph.keys())


#testing
print("Number of junctions:", len(junctions))
print("Number of roads:", sum(len(v) for v in graph.values()) // 2)
print(graph["Curepipe"])



Number of junctions: 18
Number of roads: 25
[('Forest Side', 4), ('Floreal', 6)]


# **Dijkstra's algorithm**

Dijkstra's algorithm finds the shortest travel time from one starting junction to every other junction. It keeps a priority queue of junctions ordered by their best known time. It repeatedly takes the closest unvisited junction and updates its neighbours if a faster path through it is found. It also records each junction's previous junction on the shortest path.


**This is used in two ways:**


*   Fastest route: run from Curepipe, then follow the previous-junction records
back from Pamplemousses to rebuild the actual route, not just its total time.
*   Travel-time table: run once from every junction to get the shortest travel time between every junction and every area. The coverage step uses this table.





Dijkstra works here because all travel times are positive.

# **Coverage**

A stop covers an area if the shortest travel time between the stop's junction and the area's junction is at most nine minutes. Exactly nine minutes counts as covered.


Using the travel-time table from section 2, this section builds a coverage set for each junction: the areas a stop at that junction would cover. Both the greedy method and the exhaustive search use these same sets, so they are compared on identical coverage rules.

# **Greedy stop placement**

This is the main method. It chooses six stops one at a time. At each step it adds the junction that covers the most areas not already covered by the stops chosen so far. Once a stop is chosen, it is never removed or changed.

If two or more junctions cover the same number of new areas, the one whose name comes first alphabetically is chosen. This makes the result fixed and repeatable.

Greedy is fast, but it is not guaranteed to find the best set of six. It only looks at the best choice at each step, not at how choices work together.

# **Exhaustive search (baseline)**

This is the baseline the greedy method is compared against. It tries every possible set of six junctions out of 18, which is C(18, 6) = 18,564 combinations. For each set it counts how many areas are covered, and it keeps the best.

Because it checks every option, the result is the true maximum coverage. The gap between this number and the greedy result shows how far greedy fell short on this network.

A counter records how many combinations were checked, alongside the measured running time.